# Suite RRN — Reruns and diffs: queries over the certificates

A rerun reuses the parameter assignments an earlier history recorded, and a diff compares two histories
([CAP-076: reruns reuse decisions](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-076.md)). Neither needs code of mbse-transforms': a transform certificate records its transform
(`transform`, a `References.OfObject` value), its parameter assignments (`assignments`) and its sub-certificates, each
holding the frame its argument's visitor bore (`frame`, a `References.OfFrame` value), and certificates are
mbse-schemas data. So a rerun's reuse is a query over the certificates' store: the transform certificate, in the
earlier history, of the same transform on the same frame, and the parameter assignments it recorded. A diff is the same
query over two histories, the certificates it pairs compared by mbse-schemas' `Comparison`: added, removed (the
orphans), and changed (other parameter assignments recorded). The cases use `Transforms.Session`, `Transforms.apply`
and `Transforms.certify` as any caller does, and mbse-schemas' transactions, `Plain` and `Comparison` for the rest; no
call of mbse-transforms' reads a certificate.

The tests play the caller. The fixture is suite XF's shelf: a singleton `shelf.Shelf` owning `shelf.Item`s by key,
each with a `name` and a `label`; the pattern `shelf.Unlabelled`, an item without a label, declaring the parameter
`case` (a str, left open by a session); and the transform `Label`, whose procedure labels the sync's item in the case
its parameter assignment gives. The transforms are defined in a store of their own, `transforms`, and the certificates
written in another, `certs`. A frame binds each symbol by object reference: a store remade with the same owners and
keys holds the same frames, and a history recorded over one store is queried by the frames of another.

In [ ]:
import { Builders as B, Comparison, JSON as Json, Parametrics, Plain, Proxies, References } from "@mbse/schemas/Framework";

import { Transforms } from "../src/index.js";

import { assert, bear, equal, resolve } from "./support.js";

/** shelf.Shelf, a singleton owning shelf.Items by key (shelf.Holds), each with a name and a label. */
function shelf(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("rrn.shelf/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Shelf: S.OfObject, Item: S.OfObject, Holds: S.OfRelation })
    .spec([
      (_: any) => _.Shelf.name("shelf.Shelf").ref(true).singleton("shelf.TheShelf").adjacencies({
        items: (t: any) => t.relation([_.Holds]).me("shelf") }),
      (_: any) => _.Item.name("shelf.Item").ref(true).properties({ name: S.OfStr, label: S.OfStr }).adjacencies([
        (t: any) => t.name("shelf").relation([_.Holds]).me("item").owner(true)]),
      (_: any) => _.Holds.name("shelf.Holds").links([(t: any) => t.name("shelf"), (t: any) => t.name("item")])
        .properties({ key: S.OfStr }).uniques([["shelf", "key"], ["item"]]),
    ])]))]);
}

/** shelf.Unlabelled, an item without a label, declaring the parameter case. */
function patterns(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("rrn.patterns/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Unlabelled: S.OfPattern })
    .spec([
      (_: any) => _.Unlabelled.name("shelf.Unlabelled").symbols({ i: txn.shelf.Item })
        .spec([(f: any) => f.i.label(null)])
        .parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          case: (d: any) => d.parameter((p: any) => p.description("upper or lower").for_value([S.OfStr])) }))]),
    ])]))]);
}

/** A store of the shelf's schemas and patterns, its shelf holding an item of each name, keyed by it. */
function model(...names: string[]): any {
  const store: any = new B.Store();
  store.transaction(Parametrics.install);
  store.transaction(shelf);
  store.transaction(patterns);
  store.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ s: t.shelf.Shelf })]))]));
  for (const name of names) {
    store.transaction(() => {
      store.shelf.Shelf(store.singleton("shelf.TheShelf"), (s: any) => s.items({
        [name]: (e: any) => e.item((i: any) => i.name(name)) }));
    });
  }
  return store;
}

/** Each item on the shelf: its key, name and label. */
function items(store: any): unknown[][] {
  return store.transaction(() => [...store.singleton("shelf.TheShelf").items].map((e: any) =>
    [e.key.get(), e.item.name.get(), e.item.label.get()]));
}

/** The reference of `x` (an object, or a frame's bindings), made in the store `own`, as a value of `store`'s:
 * written as plain data and read back (mbse-schemas' serialization). */
function carried(own: any, x: unknown, store: any): any {
  const reference = References.of(own, x);
  return Plain.FromPlain(store)(store.registered(reference.as_schema().name),
    Plain.ToPlain(own)(reference.as_schema(), reference));
}

/** The query a rerun and a diff make: the frames, in the certificates' store, binding each transform certificate
 * `c` in the history named `history` of the transform `transform` (a `References.OfObject` value) and its
 * sub-certificate `f` holding the frame `frame` (a `References.OfFrame` value), matched by equality in the spec. */
function recorded(store: any, history: string, transform: any, frame: any): any[] {
  const T = store.Transforms;
  return store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ d: T.Histories, h: T.HistoryCertificate, c: T.TransformCertificate, f: T.FrameCertificate })
    .spec([
      (_: any) => _.d.histories({ [history]: (e: any) => e.history(_.h) }),
      (_: any) => _.h.steps([(e: any) => e.step(_.c)]),
      (_: any) => _.c.transform(transform).frames([(e: any) => e.frame(_.f)]),
      (_: any) => _.f.frame(frame),
    ])]))]));
}

/** The query a diff starts from: the frames binding each transform certificate `c` in the history named
 * `history` and each of its sub-certificates `f`. */
function certified(store: any, history: string): any[] {
  const T = store.Transforms;
  return store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ d: T.Histories, h: T.HistoryCertificate, c: T.TransformCertificate, f: T.FrameCertificate })
    .spec([
      (_: any) => _.d.histories({ [history]: (e: any) => e.history(_.h) }),
      (_: any) => _.h.steps([(e: any) => e.step(_.c)]),
      (_: any) => _.c.frames([(e: any) => e.frame(_.f)]),
    ])]))]));
}

/** The value object a certificate's property holds, as a value of the store's, outside any transaction: written
 * as plain data and read back. */
function held(store: any, member: any): any {
  const found: any[] = [];
  member.as_object((o: any) => {
    found.push([o.as_schema(), Plain.ToPlain(store)(o.as_schema(), o)]);
    return o;
  });
  return Plain.FromPlain(store)(found[0][0], found[0][1]);
}

/** The parameter assignments a transform certificate records, read through the visitor protocol. */
function assignments_of(certificate: any): Record<string, unknown> {
  const found: Record<string, unknown> = {};
  const each = (entries: any) => {
    for (const [name, value] of entries) {
      value.as_scalar((s: any) => {
        found[name] = s.get();
        return s;
      });
    }
    return entries;
  };
  certificate.assignments.as_value((a: any) => a.as_container((c: any) => c.as_entries(each)));
  return found;
}

/** Whether two certificates, each read within the store's transaction, are equal as values of their schema
 * (mbse-schemas' `Comparison`): a transform certificate's transform, prototype and parameter assignments. */
function same(a: any, b: any): boolean {
  return new Comparison.OfObject(a.as_schema(), a).compare(new Comparison.OfObject(b.as_schema(), b)) === 0;
}

/** The key on the shelf of the item a sub-certificate's frame binds, read from it as plain data. */
function key_of(store: any, frame: any): string {
  const found: any[] = [];
  frame.as_object((o: any) => {
    found.push(Plain.ToPlain(store)(o.as_schema(), o));
    return o;
  });
  return found[0].get("binds").get("i").get("path").get("next").get("entry").get("properties").get("key");
}

/** Each transform certificate of a history: the key of the item its sub-certificate's frame binds, its
 * parameter assignments, and who decided them. */
function shown(store: any, history: string): unknown[][] {
  return store.transaction(() => {
    const held_history = store.singleton("Transforms.TheHistories").histories.at(history).history;
    const decisions = [...held_history.decisions];
    return [...held_history.steps].map((e: any, k) =>
      [key_of(store, [...e.step.frames][0].frame.frame), assignments_of(e.step), String(decisions[k].by)]);
  });
}

/** The caller's session over the data's frames of shelf.Unlabelled: each candidate of Label, its open parameter
 * answered by `decide(candidate)`, which gives the parameter assignments and who decided them; then applied, with the
 * candidate's frame as the sync, and certified into the history named `history` in the certificates' store `store`. */
function run(store: any, data: any, history: string,
  decide: (candidate: Transforms.Candidate) => [Transforms.Assignments, string]): void {
  const frames: any[] = data.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([
    data.registered("shelf.Unlabelled")]))]));
  transforms.transaction(() => data.transaction(() => {
    const Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform;
    for (const [candidate] of new Transforms.Session([Label], frames.map((f) => bear(data, "shelf.Unlabelled", f)))) {
      const [assignments, by] = decide(candidate!);
      const taken = candidate!.answer(assignments);
      Transforms.apply(Label, taken.assignments, procedures, { sync: taken.frame });
      Transforms.certify(store, Label, taken.assignments, history, by, { sync: taken.frame });
    }
  }));
}

/** The caller deciding each item's case itself, by the item's name. */
function chosen(choices: Record<string, string>): (candidate: Transforms.Candidate) => [Transforms.Assignments, string] {
  return (candidate) => [{ case: choices[String(candidate.frame.i.name)] }, "caller"];
}

const transforms: any = model(); // the transforms' store: the shelf's schemas and patterns, no data, and the transforms
transforms.transaction(Transforms.install);
transforms.transaction((t: any) => Transforms.define(t, "Label", "label", t.shelf.Unlabelled));
transforms.transaction((t: any) => Transforms.define(t, "Copy", "label", t.shelf.Unlabelled));
const certs: any = new B.Store(); // the certificates' store, as the caller gives it
certs.transaction(Transforms.install);
const procedures = new Transforms.Procedures();

/** Labels the sync's item with its name, in the case its parameter assignment gives. */
function label(source: any, sync: any, target: any, assignments: Transforms.Assignments): void {
  const name = String(sync.i.name);
  (Proxies.store_of(sync.i) as any).shelf.Item(sync.i, (i: any) => i.label(assignments.case === "upper" ? name.toUpperCase() : name.toLowerCase()));
}

procedures.register("label", label);
const [label_ref, copy_ref] = transforms.transaction(() => ["Label", "Copy"].map((n) =>
  carried(transforms, transforms.singleton("Transforms.TheCatalog").transforms.at(n).transform, certs))); // each transform's reference, as a value of the certificates' store

<a id="RRN-01"></a>
## RRN-01 · A rerun reuses: for each frame of a fresh session, a query over the certificates' store finds the transform certificate, in the earlier history, of the same transform on the same frame, and the caller applies and certifies with the parameter assignments it recorded, decided "reused"

In [ ]:
const first = model("bolt", "nut");
run(certs, first, "first", chosen({ bolt: "upper", nut: "lower" }));
assert(equal(items(first), [["bolt", "bolt", "BOLT"], ["nut", "nut", "nut"]]));

/** The caller reusing what the history named `history` recorded: for a candidate, the query for the transform
 * certificate of Label on the candidate's frame (its bindings, as certify records them, made in the data's store);
 * where it finds one, the parameter assignments it recorded, decided "reused"; where none, `otherwise`'s decision. */
function reusing(store: any, history: string, data: any,
  otherwise: (candidate: Transforms.Candidate) => [Transforms.Assignments, string]) {
  return (candidate: Transforms.Candidate): [Transforms.Assignments, string] => {
    const found = recorded(store, history, label_ref, carried(data, new Map([["i", candidate.frame.i]]), store));
    if (found.length === 0) return otherwise(candidate); // a frame the earlier history never saw
    const [given] = found;
    return store.transaction(() => [assignments_of(resolve(store, given.get("c"))), "reused"]); // read within a transaction: the certificate the frame given back binds
  };
}

const again = model("bolt", "nut", "washer"); // remade: the same owners and keys, and an item the first run never saw
run(certs, again, "second", reusing(certs, "first", again, chosen({ washer: "upper" })));
assert(equal(items(again), [["bolt", "bolt", "BOLT"], ["nut", "nut", "nut"], ["washer", "washer", "WASHER"]]));
assert(equal(shown(certs, "first"), [["bolt", { case: "upper" }, "caller"], ["nut", { case: "lower" }, "caller"]]));
assert(equal(shown(certs, "second"), [
  ["bolt", { case: "upper" }, "reused"], ["nut", { case: "lower" }, "reused"], ["washer", { case: "upper" }, "caller"]]));
certs.transaction(() => {
  const histories = certs.singleton("Transforms.TheHistories").histories;
  const [one, two] = ["first", "second"].map((h) => histories.at(h).history.steps);
  assert(same(one.at(0n).step, two.at(0n).step) && same(one.at(1n).step, two.at(1n).step)); // transform, prototype, assignments
  assert(!same(one.at(0n).step, two.at(1n).step)); // bolt's and nut's: other assignments
});

<a id="RRN-02"></a>
## RRN-02 · A diff: two histories over stores that differ, by the same query and mbse-schemas' `Comparison` alone: added, removed (the orphans), and changed (other parameter assignments recorded)

In [ ]:
const [before, after] = [model("bolt", "nut", "washer"), model("bolt", "nut", "screw")]; // washer removed, screw added
run(certs, before, "before", chosen({ bolt: "upper", nut: "upper", washer: "lower" }));
run(certs, after, "after", chosen({ bolt: "upper", nut: "lower", screw: "lower" })); // nut decided differently

/** Each transform certificate of a history, found by the query `certified`: its reference, and its transform and
 * its sub-certificate's frame, as values of the store's, to query another history by. */
function steps_of(store: any, history: string): [any, any, any][] {
  const found = certified(store, history);
  return store.transaction(() => found.map((given: any) => [given.get("c"),
    held(store, resolve(store, given.get("c")).transform), held(store, resolve(store, given.get("f")).frame)]));
}

/** What the history named `b` removes from the one named `a` (the orphans: a certificate of a's with none in b's
 * of the same transform on the same frame), what it adds (one of b's with none in a's), and what it changes (a pair
 * whose certificates are not equal as values), each by reference. */
function diff(store: any, a: string, b: string): [any[], any[], any[][]] {
  const [ours, theirs] = [steps_of(store, a), steps_of(store, b)];
  const removed = ours.filter(([, transform, frame]) => recorded(store, b, transform, frame).length === 0).map(([c]) => c);
  const added = theirs.filter(([, transform, frame]) => recorded(store, a, transform, frame).length === 0).map(([c]) => c);
  const pairs = ours.flatMap(([c, transform, frame]) => recorded(store, b, transform, frame).map((given: any) => [c, given.get("c")]));
  const changed = store.transaction(() => pairs.filter(([c, d]) => !same(resolve(store, c), resolve(store, d))));
  return [removed, added, changed];
}

/** Each transform certificate a reference designates: the key of the item its frame binds, and its parameter
 * assignments. */
function told(store: any, references: any[]): unknown[][] {
  return store.transaction(() => references.map((reference) => resolve(store, reference)).map((c: any) =>
    [key_of(store, [...c.frames][0].frame.frame), assignments_of(c)]));
}

{
  let [removed, added, changed] = diff(certs, "before", "after");
  assert(equal(told(certs, removed), [["washer", { case: "lower" }]])); // the orphan
  assert(equal(told(certs, added), [["screw", { case: "lower" }]]));
  assert(equal(changed.map((pair) => told(certs, pair)), [[["nut", { case: "upper" }], ["nut", { case: "lower" }]]])); // bolt: unchanged
  [removed, added, changed] = diff(certs, "after", "before"); // the other way: added and removed swap
  assert(equal([told(certs, removed), told(certs, added)], [[["screw", { case: "lower" }]], [["washer", { case: "lower" }]]]));
  assert(equal(diff(certs, "first", "second")[0], []) && equal(diff(certs, "first", "second")[2], [])); // a rerun that reused
}

<a id="RRN-03"></a>
## RRN-03 · What a query can match: a sub-certificate by its recorded frame, a `References.OfFrame` value, by equality in a pattern spec, as a transform certificate by its transform, a `References.OfObject` value, by its parameter assignments, and by its sub-certificate's argument; equality is of the whole value

In [ ]:
{
  const T = certs.Transforms;

  /** How many frames a map over the certificates' store gives, of the symbols and the spec given. */
  const count = (symbols: Record<string, any>, spec: ((_: any) => any)[]): number =>
    certs.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
      .symbols(symbols).spec(spec)]))])).length;

  const nut = model("nut");
  const [given] = nut.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([nut.registered("shelf.Unlabelled")]))]));
  const [witness] = nut.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ i: t.shelf.Item }).spec([(_: any) => _.i.name("nut")])]))]));
  const bound = carried(nut, given, certs); // the frame's bindings: what certify records, from a store of its own
  const renamed = nut.transaction(() => carried(nut, new Map([["j", resolve(nut, given.get("i"))]]), certs)); // the same item, bound to another symbol
  assert(count({ f: T.FrameCertificate }, [(_: any) => _.f.frame(bound)]) === 4); // nut's, in first, second, before, after
  assert(count({ f: T.FrameCertificate }, [(_: any) => _.f.frame(renamed)]) === 0); // another frame
  assert((Plain.ToPlain(nut)(References.of(nut, witness).as_schema(), References.of(nut, witness)) as any).has("assigns"));
  assert(count({ f: T.FrameCertificate }, [(_: any) => _.f.frame(carried(nut, witness, certs))]) === 0); // a witness: its values too
  assert(count({ c: T.TransformCertificate }, [(_: any) => _.c.transform(label_ref)]) === 11); // every certificate so far
  assert(count({ c: T.TransformCertificate }, [(_: any) => _.c.transform(copy_ref)]) === 0); // never applied
  for (const [argument, expected] of [["sync", 4], ["source", 0]] as const) { // the sub-certificate by its argument, the entry's key
    assert(count({ c: T.TransformCertificate, f: T.FrameCertificate }, [
      (_: any) => _.c.frames({ [argument]: (e: any) => e.frame(_.f) }), (_: any) => _.f.frame(bound)]) === expected);
  }
  const lower = [(_: any) => _.d.histories({ after: (e: any) => e.history(_.h) }), (_: any) => _.h.steps([(e: any) => e.step(_.c)]),
    (_: any) => _.c.assignments({ case: "lower" })]; // the parameter assignments, a container, by equality
  assert(count({ d: T.Histories, h: T.HistoryCertificate, c: T.TransformCertificate }, lower) === 2); // nut and screw
  assert(recorded(certs, "first", label_ref, bound).length === 1 && equal(recorded(certs, "first", copy_ref, bound), []));
}

<a id="RRN-04"></a>
## RRN-04 · Determinism: the same frames and the same choices give byte-identical certificates, as plain data, in two stores

In [ ]:
{
  /** The certificates of a run over a store of bolt and nut, deciding by `choices`, in a certificates' store of
   * their own, as plain data (mbse-schemas' snapshot), serialized. */
  const written = (choices: Record<string, string>): string => {
    const store: any = new B.Store();
    store.transaction(Transforms.install);
    run(store, model("bolt", "nut"), "run", chosen(choices));
    return Json.dumps(Plain.ToPlain(store).Snapshot());
  };

  const once = written({ bolt: "upper", nut: "lower" });
  assert(written({ bolt: "upper", nut: "lower" }) === once); // the same frames and choices: the same bytes
  assert(written({ bolt: "upper", nut: "upper" }) !== once); // other choices: other certificates
}